In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

def get_fundamentals(ticker):
    stock = yf.Ticker(ticker)
    info  = stock.info
    
    # Price & valuation
    price       = info.get('currentPrice') or info.get('regularMarketPrice')
    pe          = info.get('trailingPE')
    fwd_pe      = info.get('forwardPE')
    pb          = info.get('priceToBook')
    peg         = info.get('pegRatio')
    ev_ebitda   = info.get('enterpriseToEbitda')
    div_yield   = info.get('dividendYield')
    
    # Quality
    roe         = info.get('returnOnEquity')
    roa         = info.get('returnOnAssets')
    gross_margin= info.get('grossMargins')
    op_margin   = info.get('operatingMargins')
    net_margin  = info.get('profitMargins')
    de_ratio    = info.get('debtToEquity')
    current_r   = info.get('currentRatio')
    fcf         = info.get('freeCashflow')
    
    # Growth
    rev_growth  = info.get('revenueGrowth')
    earn_growth = info.get('earningsGrowth')
    
    # Momentum
    week52_high = info.get('fiftyTwoWeekHigh')
    week52_low  = info.get('fiftyTwoWeekLow')
    ma50        = info.get('fiftyDayAverage')
    ma200       = info.get('twoHundredDayAverage')
    
    # Company info
    name        = info.get('longName', ticker)
    sector      = info.get('sector', 'N/A')
    industry    = info.get('industry', 'N/A')
    mktcap      = info.get('marketCap')
    beta        = info.get('beta')
    
    return {
        'name': name, 'sector': sector, 'industry': industry,
        'price': price, 'mktcap': mktcap, 'beta': beta,
        'pe': pe, 'fwd_pe': fwd_pe, 'pb': pb, 'peg': peg,
        'ev_ebitda': ev_ebitda, 'div_yield': div_yield,
        'roe': roe, 'roa': roa, 'gross_margin': gross_margin,
        'op_margin': op_margin, 'net_margin': net_margin,
        'de_ratio': de_ratio, 'current_r': current_r, 'fcf': fcf,
        'rev_growth': rev_growth, 'earn_growth': earn_growth,
        'week52_high': week52_high, 'week52_low': week52_low,
        'ma50': ma50, 'ma200': ma200,
    }

# Test it
ticker = 'MA'
data = get_fundamentals(ticker)
print(f"✅ Fetched data for {data['name']}")
print(f"   Sector: {data['sector']} | Industry: {data['industry']}")
print(f"   Price: ${data['price']} | Market Cap: ${data['mktcap']:,.0f}")

✅ Fetched data for Mastercard Incorporated
   Sector: Financial Services | Industry: Credit Services
   Price: $488.645 | Market Cap: $431,758,835,712


In [2]:
def generate_signal(data):
    scores = {}
    reasons = []

    # --- VALUATION SCORE (0-100, higher = cheaper) ---
    val = 0
    count = 0
    if data['pe']:
        if data['pe'] < 15:   v=100
        elif data['pe'] < 25: v=70
        elif data['pe'] < 35: v=40
        else:                  v=15
        val += v; count += 1
        reasons.append(f"P/E of {data['pe']:.1f} — {'attractive' if v>=70 else 'fair' if v>=40 else 'stretched'}")

    if data['fwd_pe']:
        if data['fwd_pe'] < 15:   v=100
        elif data['fwd_pe'] < 25: v=70
        elif data['fwd_pe'] < 35: v=40
        else:                      v=15
        val += v; count += 1

    if data['peg']:
        if data['peg'] < 1:   v=100
        elif data['peg'] < 2: v=65
        elif data['peg'] < 3: v=35
        else:                  v=10
        val += v; count += 1
        reasons.append(f"PEG of {data['peg']:.1f} — {'growth at reasonable price' if v>=65 else 'expensive relative to growth'}")

    if data['ev_ebitda']:
        if data['ev_ebitda'] < 10:  v=100
        elif data['ev_ebitda'] < 20: v=65
        elif data['ev_ebitda'] < 30: v=35
        else:                         v=10
        val += v; count += 1

    scores['valuation'] = val / count if count else 50

    # --- QUALITY SCORE (0-100) ---
    qual = 0
    count = 0
    if data['roe']:
        if data['roe'] > 0.30:   v=100
        elif data['roe'] > 0.15: v=70
        elif data['roe'] > 0.08: v=40
        else:                     v=15
        qual += v; count += 1
        reasons.append(f"ROE of {data['roe']*100:.1f}% — {'exceptional' if v>=100 else 'solid' if v>=70 else 'weak'}")

    if data['net_margin']:
        if data['net_margin'] > 0.25:  v=100
        elif data['net_margin'] > 0.15: v=75
        elif data['net_margin'] > 0.08: v=45
        else:                            v=15
        qual += v; count += 1
        reasons.append(f"Net margin of {data['net_margin']*100:.1f}% — {'best-in-class' if v>=100 else 'healthy' if v>=75 else 'thin'}")

    if data['de_ratio']:
        if data['de_ratio'] < 50:    v=100
        elif data['de_ratio'] < 100: v=70
        elif data['de_ratio'] < 200: v=40
        else:                         v=15
        qual += v; count += 1
        reasons.append(f"D/E ratio of {data['de_ratio']:.0f} — {'conservative balance sheet' if v>=70 else 'leveraged'}")

    if data['current_r']:
        if data['current_r'] > 2:   v=100
        elif data['current_r'] > 1: v=65
        else:                        v=20
        qual += v; count += 1

    scores['quality'] = qual / count if count else 50

    # --- GROWTH SCORE (0-100) ---
    grow = 0
    count = 0
    if data['rev_growth']:
        if data['rev_growth'] > 0.20:   v=100
        elif data['rev_growth'] > 0.10: v=75
        elif data['rev_growth'] > 0.05: v=50
        elif data['rev_growth'] > 0:    v=30
        else:                            v=10
        grow += v; count += 1
        reasons.append(f"Revenue growth of {data['rev_growth']*100:.1f}% — {'strong' if v>=75 else 'moderate' if v>=50 else 'slowing'}")

    if data['earn_growth']:
        if data['earn_growth'] > 0.20:   v=100
        elif data['earn_growth'] > 0.10: v=75
        elif data['earn_growth'] > 0:    v=40
        else:                             v=10
        grow += v; count += 1
        reasons.append(f"Earnings growth of {data['earn_growth']*100:.1f}% — {'accelerating' if v>=75 else 'modest' if v>=40 else 'declining'}")

    scores['growth'] = grow / count if count else 50

    # --- MOMENTUM SCORE (0-100) ---
    mom = 0
    count = 0
    if data['price'] and data['week52_high'] and data['week52_low']:
        range_pos = (data['price'] - data['week52_low']) / (data['week52_high'] - data['week52_low'])
        if range_pos > 0.75:   v=90
        elif range_pos > 0.50: v=65
        elif range_pos > 0.25: v=40
        else:                   v=20
        mom += v; count += 1
        reasons.append(f"Trading at {range_pos*100:.0f}% of 52-week range — {'near highs' if v>=90 else 'mid-range' if v>=65 else 'near lows'}")

    if data['price'] and data['ma50'] and data['ma200']:
        if data['price'] > data['ma50'] > data['ma200']:   v=100  # golden cross
        elif data['price'] > data['ma200']:                 v=65
        elif data['price'] > data['ma50']:                  v=50
        else:                                               v=20
        mom += v; count += 1
        reasons.append(f"Price vs MAs — {'bullish trend' if v>=100 else 'above 200MA' if v>=65 else 'weak trend'}")

    scores['momentum'] = mom / count if count else 50

    # --- COMPOSITE & SIGNAL ---
    composite = (
        scores['valuation'] * 0.30 +
        scores['quality']   * 0.30 +
        scores['growth']    * 0.20 +
        scores['momentum']  * 0.20
    )

    if composite >= 65:   signal, color = 'BUY',  '🟢'
    elif composite >= 45: signal, color = 'HOLD', '🟡'
    else:                 signal, color = 'SELL', '🔴'

    return scores, composite, signal, color, reasons

scores, composite, signal, color, reasons = generate_signal(data)
print(f"✅ Signal generated: {color} {signal} (score: {composite:.1f}/100)")

✅ Signal generated: 🟡 HOLD (score: 54.9/100)


In [5]:
def print_brief(ticker, data, scores, composite, signal, color, reasons):
    mktcap_b = f"${data['mktcap']/1e9:.1f}B" if data['mktcap'] else 'N/A'
    
    def fmt_pct(v): return f"{v*100:.1f}%" if v else 'N/A'
    def fmt_x(v):   return f"{v:.1f}x" if v else 'N/A'
    def fmt_n(v):   return f"{v:.1f}" if v else 'N/A'
    def fmt_fcf(v): return f"${v/1e9:.1f}B" if v else 'N/A'
    def fmt_div(v):
        if not v: return 'None'
        return f"{v*100:.1f}%" if v < 0.5 else f"{v:.1f}%"

    div = fmt_div(data['div_yield'])

    print(f"""
{'='*60}
  INVESTMENT BRIEF — {datetime.today().strftime('%B %d, %Y')}
{'='*60}
  {data['name']} ({ticker.upper()})
  {data['sector']} | {data['industry']}
  Price: ${data['price']:.2f}  |  Mkt Cap: {mktcap_b}  |  Beta: {fmt_n(data['beta'])}
  Dividend Yield: {div}
{'='*60}

  SIGNAL: {color} {signal}  ({composite:.1f} / 100)

{'─'*60}
  SCORES
{'─'*60}
  Valuation  : {scores['valuation']:.1f}/100  {'█' * int(scores['valuation']//10)}
  Quality    : {scores['quality']:.1f}/100  {'█' * int(scores['quality']//10)}
  Growth     : {scores['growth']:.1f}/100  {'█' * int(scores['growth']//10)}
  Momentum   : {scores['momentum']:.1f}/100  {'█' * int(scores['momentum']//10)}

{'─'*60}
  VALUATION
{'─'*60}
  Trailing P/E   : {fmt_x(data['pe'])}
  Forward P/E    : {fmt_x(data['fwd_pe'])}
  PEG Ratio      : {fmt_n(data['peg'])}
  EV/EBITDA      : {fmt_x(data['ev_ebitda'])}
  Price/Book     : {fmt_x(data['pb'])}

{'─'*60}
  QUALITY
{'─'*60}
  ROE            : {fmt_pct(data['roe'])}
  ROA            : {fmt_pct(data['roa'])}
  Gross Margin   : {fmt_pct(data['gross_margin'])}
  Operating Margin: {fmt_pct(data['op_margin'])}
  Net Margin     : {fmt_pct(data['net_margin'])}
  Debt/Equity    : {fmt_n(data['de_ratio'])}
  Current Ratio  : {fmt_n(data['current_r'])}
  Free Cash Flow : {fmt_fcf(data['fcf'])}

{'─'*60}
  GROWTH
{'─'*60}
  Revenue Growth : {fmt_pct(data['rev_growth'])}
  Earnings Growth: {fmt_pct(data['earn_growth'])}

{'─'*60}
  MOMENTUM
{'─'*60}
  52W High       : ${data['week52_high']:.2f}
  52W Low        : ${data['week52_low']:.2f}
  50D MA         : ${data['ma50']:.2f}
  200D MA        : ${data['ma200']:.2f}
  vs 52W High    : {((data['price']/data['week52_high'])-1)*100:.1f}%

{'─'*60}
  KEY OBSERVATIONS
{'─'*60}""")
    for r in reasons:
        print(f"  • {r}")
    print(f"""
{'─'*60}
  ⚠️  This is a quantitative signal only. Always do your
  own due diligence before making investment decisions.
{'='*60}
""")

# Run it
print_brief(ticker, data, scores, composite, signal, color, reasons)


  INVESTMENT BRIEF — June 12, 2026
  Mastercard Incorporated (MA)
  Financial Services | Credit Services
  Price: $488.64  |  Mkt Cap: $431.8B  |  Beta: 0.7
  Dividend Yield: 0.7%

  SIGNAL: 🟡 HOLD  (54.9 / 100)

────────────────────────────────────────────────────────────
  SCORES
────────────────────────────────────────────────────────────
  Valuation  : 52.5/100  █████
  Quality    : 58.8/100  █████
  Growth     : 87.5/100  ████████
  Momentum   : 20.0/100  ██

────────────────────────────────────────────────────────────
  VALUATION
────────────────────────────────────────────────────────────
  Trailing P/E   : 28.3x
  Forward P/E    : 21.5x
  PEG Ratio      : 1.5
  EV/EBITDA      : 20.6x
  Price/Book     : 64.5x

────────────────────────────────────────────────────────────
  QUALITY
────────────────────────────────────────────────────────────
  ROE            : 232.1%
  ROA            : 25.0%
  Gross Margin   : 100.0%
  Operating Margin: 60.8%
  Net Margin     : 45.9%
  Debt/Equit

In [6]:
def analyze(ticker):
    ticker = ticker.strip().upper()
    print(f"Fetching data for {ticker}...")
    try:
        data = get_fundamentals(ticker)
        scores, composite, signal, color, reasons = generate_signal(data)
        print_brief(ticker, data, scores, composite, signal, color, reasons)
    except Exception as e:
        print(f"❌ Could not fetch data for {ticker}. Check the ticker and try again. ({e})")

# ── CHANGE THIS TICKER ANYTIME ──
analyze('NVDA')

Fetching data for NVDA...

  INVESTMENT BRIEF — June 12, 2026
  NVIDIA Corporation (NVDA)
  Technology | Semiconductors
  Price: $204.81  |  Mkt Cap: $4960.7B  |  Beta: 2.2
  Dividend Yield: 49.0%

  SIGNAL: 🟢 BUY  (81.4 / 100)

────────────────────────────────────────────────────────────
  SCORES
────────────────────────────────────────────────────────────
  Valuation  : 61.2/100  ██████
  Quality    : 100.0/100  ██████████
  Growth     : 100.0/100  ██████████
  Momentum   : 65.0/100  ██████

────────────────────────────────────────────────────────────
  VALUATION
────────────────────────────────────────────────────────────
  Trailing P/E   : 31.3x
  Forward P/E    : 16.1x
  PEG Ratio      : 0.6
  EV/EBITDA      : 29.7x
  Price/Book     : 25.4x

────────────────────────────────────────────────────────────
  QUALITY
────────────────────────────────────────────────────────────
  ROE            : 114.3%
  ROA            : 52.7%
  Gross Margin   : 74.1%
  Operating Margin: 65.6%
  Net Mar

# 📊 Stock Investment Brief Tool

## Overview
A Python-based fundamental analysis tool that generates a one-page investment brief 
for any stock or ETF. Input a ticker, get a structured Buy/Hold/Sell signal backed 
by real market data and quantitative scoring.

---

## How It Works

### Step 1 — Data Fetch
Pulls ~25 live data points from Yahoo Finance across four dimensions:
- **Valuation** — P/E, Forward P/E, PEG, EV/EBITDA, Price/Book
- **Quality** — ROE, ROA, Gross/Operating/Net Margins, Debt/Equity, Current Ratio, FCF
- **Growth** — Revenue Growth, Earnings Growth
- **Momentum** — 52-week range position, 50D & 200D moving averages

### Step 2 — Scoring
Each metric is converted to a 0–100 score using institutional thresholds:

| Metric     | Score 100      | Score 70       | Score 40       | Score 15    |
|------------|----------------|----------------|----------------|-------------|
| P/E        | < 15           | 15–25          | 25–35          | > 35        |
| ROE        | > 30%          | 15–30%         | 8–15%          | < 8%        |
| Net Margin | > 25%          | 15–25%         | 8–15%          | < 8%        |
| PEG        | < 1            | 1–2            | 2–3            | > 3         |
| D/E Ratio  | < 50           | 50–100         | 100–200        | > 200       |

Scores are averaged within each category, then combined into a composite:

Composite = Valuation×30% + Quality×30% + Growth×20% + Momentum×20%

### Step 3 — Signal
| Composite Score | Signal       |
|-----------------|--------------|
| 65 – 100        | 🟢 BUY       |
| 45 – 64         | 🟡 HOLD      |
| 0 – 44          | 🔴 SELL      |

---

## Why These Weights?

**Valuation & Quality (30% each)** — Most reliable long-term return predictors, 
consistent with academic factor research and CFA curriculum frameworks.

**Growth (20%)** — Matters but harder to predict consistently; used as a 
supporting factor rather than a primary driver.

**Momentum (20%)** — Price trends carry information. The market is often right 
about direction even when wrong about magnitude.

---

## Example Output — Mastercard (MA)

| Category   | Score    | Insight                                      |
|------------|----------|----------------------------------------------|
| Valuation  | 52.5/100 | Fair — P/E 28x, EV/EBITDA 20x               |
| Quality    | 58.8/100 | Strong — ROE 232%, Net Margin 45.9%, FCF $16B|
| Growth     | 87.5/100 | Excellent — Revenue +15.8%, Earnings +21.2%  |
| Momentum   | 20.0/100 | Weak — 18.8% off 52W high, below 200D MA     |

**Signal: 🟡 HOLD (54.9/100)**

> Great business at a fair price, but price trend is working against entry. 
> Momentum needs to stabilize before initiating a position.

---

## How to Use

```python
# Analyze any ticker
analyze('AAPL')
analyze('NVDA')
analyze('VNQ')
```

---

## Tech Stack
- `yfinance` — live market & fundamental data
- `pandas` / `numpy` — data processing & scoring
- `datetime` — dynamic report dating

---

## Disclaimer
This tool generates quantitative signals only. Always conduct independent 
due diligence before making investment decisions. Not financial advice.

In [7]:
analyze('SPCX')

Fetching data for SPCX...

  INVESTMENT BRIEF — June 12, 2026
  Space Exploration Technologies Corp. Class A Common Stock (SPCX)
  Industrials | Aerospace & Defense
  Price: $160.53  |  Mkt Cap: $2099.1B  |  Beta: N/A
  Dividend Yield: None

  SIGNAL: 🟡 HOLD  (59.5 / 100)

────────────────────────────────────────────────────────────
  SCORES
────────────────────────────────────────────────────────────
  Valuation  : 55.0/100  █████
  Quality    : 50.0/100  █████
  Growth     : 75.0/100  ███████
  Momentum   : 65.0/100  ██████

────────────────────────────────────────────────────────────
  VALUATION
────────────────────────────────────────────────────────────
  Trailing P/E   : N/A
  Forward P/E    : -1783.7x
  PEG Ratio      : N/A
  EV/EBITDA      : 201.7x
  Price/Book     : 27.0x

────────────────────────────────────────────────────────────
  QUALITY
────────────────────────────────────────────────────────────
  ROE            : N/A
  ROA            : N/A
  Gross Margin   : 48.8%
  Op